# Notebook_A (Sinh viên A): dữ liệu, EDA, làm sạch, SQL, hình RQ1

**Đề tài**: Nhu cầu chăm sóc cây xanh San Francisco theo tuần (DataSF 311 và kiểm kê cây)

**Khung SDG**: SDG 11 và 15: "urban green spaces"; tiêu đề, keyword và abstract của bài dùng đúng cụm từ này.

**Vai trò**: Sinh viên A làm Bước 0 đến 3 và phần hình RQ1 của Bước 5, bàn giao `data/processed/feat.parquet` cùng `manifest.json` cho Sinh viên B (Notebook_B). Chạy tuần tự trong VS Code với extension Python và Jupyter, interpreter `.venv`. Mỗi bước có ô hướng dẫn *làm gì, vì sao, đọc thế nào* rồi đến ô code; mọi bảng lưu vào `report/` với tên `table_*.csv`, mọi hình `fig_*.png`.

**Quy tắc**: seed 42; không xoá dòng mà không ghi lý do; mọi prompt AI ghi vào Audit Log cá nhân; báo cáo slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45) theo mẫu ở ô cuối.

## Vừa làm vừa hỏi AI và ghi Audit Log: cách làm nhẹ nhàng, không rối

**Tinh thần**: AI là bạn cùng làm, không phải người làm thay. Bạn được hỏi AI bất cứ lúc nào; điều duy nhất phải giữ là *kiểm tra một lần* trước khi dùng và *ghi lại 2 phút* nếu câu hỏi đó thay đổi việc bạn làm.

**Khi nào hỏi AI** (theo thứ tự):
1. Đọc ô hướng dẫn của bước đang làm (1 phút).
2. Thử tự làm 15 phút.
3. Bí quá 15 phút thì hỏi AI với mẫu: *"Tôi đang làm Bước X của dự án Y. Dữ liệu có cột A, B, C. Tôi muốn Z. Đây là code và lỗi: ... Hãy giải thích nguyên nhân và sửa, giữ nguyên tên biến."*
4. Chạy thử code AI đưa trên dữ liệu thật; so một con số bằng tay hoặc bằng cách thứ hai.
5. Nếu vẫn bí sau 30 phút, ghi lại lỗi và hỏi giảng viên ở kênh nhóm.

**Ghi Audit Log thế nào cho không áp lực**: chỉ ghi các prompt *đã thay đổi việc bạn làm* (thường 3 đến 5 prompt một tuần, 15 đến 20 cả kỳ). Mỗi dòng 5 ô, điền trong 2 phút ngay khi dùng, không để cuối tuần:

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì (1 dòng) | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 09/09, Bước 3 | viết truy vấn trung bình trượt 7 ngày theo trạm | đưa AVG OVER ROWS 6 PRECEDING | thiếu PARTITION BY site_num, thêm và thử 2 trạm | Q3 trong sql/queries.sql |

Cột thứ tư chính là *Human Delta* và là thứ hội đồng hỏi; ghi thật, kể cả khi AI đúng ("kiểm tra bằng 10 dòng tính tay, khớp"). Ba lần bạn phát hiện AI sai trong cả kỳ là đủ yêu cầu.

**Nhịp làm việc**: mỗi ngày 1,5 đến 2 giờ vào đúng bước của tuần, xong một ô thì commit; thứ Ba và thứ Sáu slot 1 báo cáo năm mục đúng tiến độ, dù kết quả chưa đẹp. Siêng và đúng nhịp quan trọng hơn giỏi: nhóm báo cáo đều 6 lần trong 3 tuần luôn có bài, nhóm im lặng đến tuần 3 thường không kịp.

## Bước 0: môi trường và cấu trúc thư mục

Chạy một lần trong Terminal của VS Code, không phải trong ô Python. Sau khi cài xong chọn interpreter `.venv` (Ctrl+Shift+P, *Python: Select Interpreter*).

```bash
# Step 1a: environment and project layout (run once)
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost catboost \
    statsmodels shap matplotlib seaborn ydata-profiling mapie jupyter
pip freeze > requirements.txt
mkdir -p data/raw data/processed notebooks sql src report
printf "data/raw\ndata/processed\n.venv\n*.parquet\n" > .gitignore
```

In [1]:
# Environment check and shared imports for the whole notebook
import truststore
truststore.inject_into_ssl()
import pandas as pd, numpy as np, duckdb, matplotlib
import matplotlib.pyplot as plt, pathlib, warnings
warnings.filterwarnings('ignore')
for d in ['data/raw', 'data/processed', 'notebooks', 'sql', 'src', 'report']: pathlib.Path(d).mkdir(parents=True, exist_ok=True)
np.random.seed(42); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 2.3.3 | duckdb 1.5.5


## Vì sao sáu bước và mỗi bước giúp gì cho bài

| Bước | Mục đích | Lợi ích cho bài | Bằng chứng, câu hỏi |
|---|---|---|---|
| 1 Nạp, hiểu dữ liệu | biết quy mô, thời gian, đơn vị, mục tiêu | bảng mô tả dữ liệu, phạm vi hợp lệ | table_describe, profile; nền mọi RQ |
| 2 Làm sạch, ghép | dữ liệu tin được, có nguồn thứ hai | nhật ký làm sạch trả lời phản biện; đặc trưng ngoại sinh | cleaning_log, tỷ lệ khớp; RQ1, RQ2 |
| 3 SQL, EDA | trả lời câu hỏi mô tả bằng số có kiểm định; đặc trưng đúng thời điểm | bảng RQ1 có p-value; không rò rỉ; điểm SQL | table_q*, rq1a-c; RQ1 |
| 4 Chia dữ liệu | đánh giá công bằng | mốc naive cho biết cải thiện thật | table_features; RQ2 |
| 5 Trực quan hoá | thấy cấu trúc và bất thường; truyền đạt | hình có caption kết luận | fig_rq1_*; RQ1 |
| 6 Mô hình, RQ3 | so năm mô hình công bằng; đóng góp chính | bảng có độ lệch, tinh chỉnh có căn cứ, SHAP | table_rq2, rq3; RQ2, RQ3 |

**Vì sao phải chọn cột từ nhiều cột**: cột thừa làm mô hình học nhiễu, cột chứa giá trị tương lai làm kết quả đẹp giả, cột không có lúc vận hành làm mô hình vô dụng. Bốn bước giúp chọn đúng cột và khớp RQ: (1) tài liệu cơ quan cho biết ý nghĩa vật lý; (2) bảng thiếu và bảng tương quan trong EDA loại cột thiếu nhiều hoặc trùng; (3) SQL với LAG/LEAD tách quá khứ khỏi tương lai; (4) SHAP sau mô hình xác nhận cột thực sự đóng góp. Mỗi cột giữ lại phải nối về một RQ: cột mô tả cho RQ1, cột dự báo cho RQ2, cột nhóm hoặc điều kiện cho RQ3.

## Bước 1: hiểu bài toán và nạp dữ liệu

### 1.1. Bài toán và ba câu hỏi

Ghi lại ở đây ba câu hỏi nghiên cứu của đề cương (RQ1 mô tả, RQ2 mô hình, RQ3 câu hỏi thứ ba) và biến mục tiêu; mọi bảng và hình sau này phải nối về một trong ba câu hỏi.

**Làm gì**: tải file gốc từ trang cơ quan vào `data/raw/`, nạp bằng DuckDB (đọc thẳng file từ đĩa, chỉ lấy cột và dòng cần) rồi in số dòng, số cột, khoảng thời gian, số đơn vị.

**Kiểm tra**: `df.shape` khớp tài liệu cơ quan; khoảng thời gian đúng; số đơn vị đúng phạm vi đề cương.

In [2]:
# Step 1b: San Francisco 311 tree maintenance cases (DataSF Socrata API) and the Street Tree List
import pandas as pd
cases = pd.read_csv("https://data.sfgov.org/resource/vw6y-z8j6.csv?$where=service_name='Tree%20Maintenance'&$limit=500000", low_memory=False)
cases['opened'] = pd.to_datetime(cases.requested_datetime); cases['date'] = cases.opened.dt.floor('D')
cases['nbhd'] = cases['analysis_neighborhood']
trees = pd.read_csv('https://data.sfgov.org/resource/tkzw-k3nq.csv?$limit=300000', low_memory=False)      # street tree list: species, plant date, dbh, location
print(cases.shape, trees.shape, cases.date.min(), cases.date.max())

(174999, 28) (144458, 27) 2008-07-01 00:00:00 2026-09-12 00:00:00


### 1.1b. Hồ sơ tự động

`ydata-profiling` trên mẫu 20 nghìn dòng tạo `report/profile.html`; mở bằng trình duyệt để xem nhanh phân phối, thiếu, tương quan trước khi tự viết EDA ở các ô sau.

In [3]:
from ydata_profiling import ProfileReport
sample = cases.sample(min(20000, len(cases)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
cases.describe(include='all').T.to_csv('report/table_describe_raw.csv')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 28/28 [00:01<00:00, 17.57it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Bước 2: hiểu dữ liệu và làm sạch, ghép nguồn thứ hai

**Làm gì**: chuẩn tên cột; bỏ trùng theo khoá; bỏ thiếu ở biến mục tiêu; loại giá trị ngoài khoảng vật lý theo tài liệu cơ quan; dựng khoá thời gian đầy đủ và nội suy ngắn; ghép nguồn thứ hai; lưu parquet.

**Kiểm tra**: số dòng sau merge không tăng; tỷ lệ khớp trên 90%; ghi số dòng bỏ ở mỗi bước vào nhật ký làm sạch (ô sau).

In [4]:
# Tải dữ liệu thời tiết NOAA GHCN-Daily cho trạm SFO (USW00023272)
# Lưu ý: trạm này không báo cáo dữ liệu gió (AWND/WSF2/WSFG đều rỗng) trong dataset daily-summaries
# nên chỉ lấy mưa (PRCP) và nhiệt độ (TMAX); cột wsf2 giữ lại nhưng để trống để không phải sửa code các bước sau
import pandas as pd

url = (
    "https://www.ncei.noaa.gov/access/services/data/v1"
    "?dataset=daily-summaries"
    "&stations=USW00023272"
    "&dataTypes=PRCP,TMAX"
    "&startDate=2015-01-01"
    "&endDate=2026-09-13"
    "&format=csv"
    "&units=standard"
)
wx_raw = pd.read_csv(url)
wx = wx_raw.rename(columns={'DATE': 'date', 'PRCP': 'prcp', 'TMAX': 'tmax'})[['date', 'prcp', 'tmax']]
wx['wsf2'] = pd.NA
wx['date'] = pd.to_datetime(wx['date'])
wx.to_csv('data/raw/sf_daily_weather.csv', index=False)
print('Đã lưu data/raw/sf_daily_weather.csv (không có dữ liệu gió, trạm SFO không báo cáo):', wx.shape)
print(wx.notna().sum())

Đã lưu data/raw/sf_daily_weather.csv (không có dữ liệu gió, trạm SFO không báo cáo): (4235, 4)
date    4235
prcp    4235
tmax    4213
wsf2       0
dtype: int64


In [5]:
# Step 2: daily counts per neighborhood 2015-2025 with tree inventory features and weather (about 100 thousand neighborhood-days)
cases = cases[(cases.date >= '2015-01-01')].dropna(subset=['nbhd'])
cases['storm_kw'] = cases.service_details.astype(str).str.contains('fall|down|hazard|branch|broken', case=False).astype(int)
d = cases.groupby(['nbhd','date']).agg(n=('service_request_id','count'), hazard=('storm_kw','sum')).reset_index()
full = d.set_index('date').groupby('nbhd')[['n','hazard']].resample('D').sum().reset_index()

trees = trees.rename(columns={'mapdbh': 'dbh'}).dropna(subset=['dbh'])
trees['dbh'] = pd.to_numeric(trees.dbh, errors='coerce')
inv = trees.groupby('analysis_neighborhood').agg(n_trees=('treeid','count'), dbh_mean=('dbh','mean')).reset_index()

wx = pd.read_csv('data/raw/sf_daily_weather.csv', parse_dates=['date'])          # date, prcp, wsf2 (max gust), tmax from NOAA GHCN USW00023272
full = full.merge(wx, on='date', how='left')
full.to_parquet('data/processed/sf_trees.parquet', index=False); inv.to_parquet('data/processed/sf_tree_inventory.parquet', index=False); print(full.shape, full.nbhd.nunique())

(174413, 7) 41


### 2.1. Nhật ký làm sạch (bảng đưa vào bài)

Mỗi bước: tên bước, số dòng trước, số dòng sau, số bỏ, lý do có tài liệu.

In [6]:
# Re-run the cleaning steps as functions so each one is logged (keep the same order as the cell above)
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d); d = fn(d); rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])
_, cleaning_log = clean_log(full, [
    ('drop duplicates', lambda x: x.drop_duplicates(['nbhd', 'date']), 'same unit and timestamp'),
    ('drop missing target', lambda x: x.dropna(subset=['n']), 'cannot be forecast'),
])
cleaning_log.to_csv('report/table_cleaning_log.csv', index=False); print(cleaning_log)

                  step  rows_before  rows_after  dropped                   reason
0      drop duplicates       174413      174413        0  same unit and timestamp
1  drop missing target       174413      174413        0       cannot be forecast


### 1.2. EDA phần 1: cấu trúc bảng (info, describe) và cách đọc

**Làm gì**: `info()` cho biết kiểu dữ liệu và số ô không thiếu của từng cột; `describe()` cho thống kê năm số và trung bình; với cột phân loại xem số giá trị khác nhau và giá trị phổ biến nhất.

**Đọc thế nào**: cột số mà `dtype` là object nghĩa là có ký tự lạ cần ép kiểu; `max` bất thường (ví dụ 999, -9999) là mã thiếu của cơ quan; `std` bằng 0 là cột hằng, bỏ. Ghi lại các phát hiện này vào bảng `report/table_eda_notes.csv` để đưa vào mục Data của bài.

In [7]:
eda = full.copy()
import io
buf = io.StringIO(); eda.info(buf=buf); print(buf.getvalue()[:3000])
desc_num = eda.describe().T                         # numeric statistics
desc_num['skew'] = eda.select_dtypes('number').skew()  # skewness, > 1 means strongly right-skewed
desc_num.round(3).to_csv('report/table_describe_numeric.csv'); print(desc_num.round(3).head(15))
desc_cat = eda.select_dtypes(exclude='number').describe().T   # categorical columns: count, unique, top, freq
desc_cat.to_csv('report/table_describe_categorical.csv'); print(desc_cat.head(15))
notes = []                                          # record EDA notes for the paper
for c in eda.select_dtypes('number').columns:
    s = eda[c]
    if s.std() == 0: notes.append([c, 'constant column, drop'])
    if s.max() in (999, 9999, 99999, -9999): notes.append([c, 'agency missing code, replace with NaN'])
    if abs(s.skew()) > 2: notes.append([c, 'strongly skewed, consider log or tree models'])
pd.DataFrame(notes, columns=['column', 'note']).to_csv('report/table_eda_notes.csv', index=False); print(notes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174413 entries, 0 to 174412
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   nbhd    174413 non-null  object        
 1   date    174413 non-null  datetime64[ns]
 2   n       174413 non-null  int64         
 3   hazard  174413 non-null  int64         
 4   prcp    172895 non-null  float64       
 5   tmax    171993 non-null  float64       
 6   wsf2    0 non-null       float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(1)
memory usage: 9.3+ MB



           count                           mean                  min                  25%                  50%                  75%                  max  \
date      174413  2020-11-09 18:42:17.439296512  2015-01-01 00:00:00  2017-12-12 00:00:00  2020-11-10 00:00:00  2023-10-09 00:00:00  2026-09-12 00:00:00   
n       174413.0                       0.749921                  0.0                  0.0                  0.0                  1.0                 86.0   
hazard  174413.0                       0.118076                  0.0                  0.0                  0.0                  0.0                 62.0   
prcp    172895.0                        0.05706                  0.0                  0.0                  0.0                  0.0                 5.46   
tmax    171993.0                      64.408772                 46.0                 60.0                 64.0                 68.0                106.0   
wsf2         0.0                            NaN                 

In [8]:
notes_df = pd.read_csv('report/table_eda_notes.csv')
notes_df = pd.concat([notes_df, pd.DataFrame([
    {'column': 'wsf2', 'note': 'trạm SFO (USW00023272) không báo cáo dữ liệu gió trong NOAA daily-summaries; cột để trống có chủ đích, không dùng làm đặc trưng gió'}
])], ignore_index=True)
notes_df.to_csv('report/table_eda_notes.csv', index=False)
print(notes_df)

   column                                               note
0       n       strongly skewed, consider log or tree models
1  hazard       strongly skewed, consider log or tree models
2    prcp       strongly skewed, consider log or tree models
3    wsf2  trạm SFO (USW00023272) không báo cáo dữ liệu g...


### 1.3. EDA phần 2: thiếu dữ liệu, mẫu thiếu và quyết định xử lý

**Cách xem**: tỷ lệ thiếu từng cột; thiếu theo thời gian (có tháng nào mất hẳn không); thiếu theo đơn vị (trạm, pin, bang); thiếu có đi cùng nhau không (ma trận đồng thiếu).

**Quyết định xử lý** (ghi vào bảng và vào bài):
- Thiếu ở **biến mục tiêu**: bỏ dòng, không bao giờ điền.
- Thiếu **ngắn** trong chuỗi thời gian (1-3 bước): nội suy tuyến tính theo từng đơn vị.
- Thiếu **dài** hoặc cả tháng: bỏ đoạn đó của đơn vị, ghi lý do.
- Cột thiếu trên 40%: bỏ cột hoặc chỉ giữ cờ có/không.
- Cột phân loại thiếu: gộp vào lớp `unknown`.
- Với mô hình cây (LightGBM, XGBoost) thiếu trong đặc trưng có thể để nguyên, mô hình tự xử lý; với ridge phải điền (median) sau khi chia train/test.

In [9]:
miss = eda.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
miss['decision'] = np.select([miss.missing_rate == 0, miss.missing_rate < 0.05, miss.missing_rate < 0.4],
                             ['keep', 'interpolate/impute median', 'drop or keep flag'], 'drop column')
miss.round(4).to_csv('report/table_missing.csv'); print(miss.head(15))
# missingness of the target by time and by unit
tcol, gcol = 'date', 'nbhd'
by_time = eda.groupby(pd.to_datetime(eda[tcol], errors='coerce').dt.to_period('M') if not np.issubdtype(eda[tcol].dtype, np.number) else eda[tcol])['n'].apply(lambda s: s.isna().mean())
by_unit = eda.groupby(gcol)['n'].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
print('target missing by period (top):', by_time.sort_values(ascending=False).head(5).round(3).to_dict())
print('target missing by unit (top):', by_unit.head(5).round(3).to_dict())
# co-missingness matrix: do columns go missing together
co = eda.isna().astype(int); co = co.loc[:, co.sum() > 0]
if co.shape[1] > 1: print(co.corr().round(2))
fig, ax = plt.subplots(figsize=(7, 3))
miss.missing_rate.head(12).plot.bar(ax=ax, color='#1B6B6D'); ax.set_ylabel('Missing rate'); ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig('report/fig_eda_missing.png', dpi=300)

        missing_rate                   decision
wsf2        1.000000                drop column
tmax        0.013875  interpolate/impute median
prcp        0.008703  interpolate/impute median
n           0.000000                       keep
date        0.000000                       keep
nbhd        0.000000                       keep
hazard      0.000000                       keep
target missing by period (top): {Period('2015-01', 'M'): 0.0, Period('2015-02', 'M'): 0.0, Period('2015-03', 'M'): 0.0, Period('2015-04', 'M'): 0.0, Period('2015-05', 'M'): 0.0}
target missing by unit (top): {'Bayview Hunters Point': 0.0, 'Bernal Heights': 0.0, 'Castro/Upper Market': 0.0, 'Chinatown': 0.0, 'Excelsior': 0.0}
      prcp  tmax  wsf2
prcp  1.00  0.79   NaN
tmax  0.79  1.00   NaN
wsf2   NaN   NaN   NaN


### 1.4. EDA phần 3: chọn cột và thống kê theo nhóm, theo thời gian

**Chọn cột** theo ba tiêu chí: (1) có ý nghĩa vật lý hoặc nghiệp vụ với mục tiêu; (2) có sẵn tại thời điểm dự báo (không phải giá trị tương lai); (3) không trùng lặp với cột khác (tương quan trên 0,95 thì giữ một). Bảng `table_columns.csv` ghi cột nào giữ, cột nào bỏ và vì sao; bảng này là một phần của mục Data trong bài.

**Thống kê**: trung bình, trung vị, phân vị 5 và 95 của mục tiêu theo nhóm và theo thời kỳ; đây là bảng đầu tiên trả lời RQ1 trước khi làm SQL.

In [10]:
num_cols = [c for c in ['n', 'hazard', 'prcp', 'wsf2', 'tmax'] if c in eda.columns]
corr = eda[num_cols].corr()
corr.round(3).to_csv('report/table_corr.csv'); print(corr.round(3))
cols = []
for c in eda.columns:
    if c == 'n': cols.append([c, 'target', 'keep']); continue
    if c in ('date', 'nbhd'): cols.append([c, 'time/unit key', 'keep for feature building']); continue
    if c in num_cols:
        r = corr.loc[c, 'n'] if 'n' in corr.columns else np.nan
        cols.append([c, f'correlation with target {r:.2f}', 'keep' if abs(r) > 0.05 or np.isnan(r) else 'consider dropping'])
    else: cols.append([c, 'other', 'review'])
pd.DataFrame(cols, columns=['column', 'reason', 'decision']).to_csv('report/table_columns.csv', index=False)
# target statistics by unit and by period
q = lambda s: pd.Series({'mean': s.mean(), 'median': s.median(), 'p05': s.quantile(0.05), 'p95': s.quantile(0.95), 'n': s.count()})
by_g = eda.groupby('nbhd')['n'].apply(q).unstack().sort_values('mean', ascending=False)
by_g.round(3).to_csv('report/table_target_by_unit.csv'); print(by_g.head(10).round(3))
period = eda['date'] if np.issubdtype(eda['date'].dtype, np.number) else pd.to_datetime(eda['date'], errors='coerce').dt.month
by_t = eda.groupby(period)['n'].apply(q).unstack()
by_t.round(3).to_csv('report/table_target_by_period.csv'); print(by_t.round(3))

            n  hazard   prcp  wsf2   tmax
n       1.000   0.572  0.099   NaN  0.012
hazard  0.572   1.000  0.180   NaN -0.043
prcp    0.099   0.180  1.000   NaN -0.202
wsf2      NaN     NaN    NaN   NaN    NaN
tmax    0.012  -0.043 -0.202   NaN  1.000
                        mean  median  p05  p95       n
nbhd                                                  
Mission                3.482     3.0  0.0  9.0  4271.0
West of Twin Peaks     1.950     1.0  0.0  6.0  4271.0
Bayview Hunters Point  1.696     1.0  0.0  5.0  4272.0
Sunset/Parkside        1.521     1.0  0.0  4.0  4269.0
Castro/Upper Market    1.455     1.0  0.0  4.0  4271.0
Bernal Heights         1.289     1.0  0.0  4.0  4270.0
Hayes Valley           1.217     1.0  0.0  4.0  4272.0
South of Market        1.149     1.0  0.0  4.0  4272.0
Outer Richmond         1.061     1.0  0.0  3.0  4271.0
Noe Valley             1.058     1.0  0.0  4.0  4271.0
       mean  median  p05  p95        n
date                                  
1     0.75

## Bước 3: phân tích bằng SQL, trả lời RQ1 và tạo đặc trưng

**Làm gì**: đưa parquet vào DuckDB; Q1 và Q2 tổng hợp trả lời RQ1 (GROUP BY, window, CTE); Q3 tạo đặc trưng trễ, trung bình trượt và mục tiêu tương lai bằng LAG, LEAD, AVG OVER; Q4 kiểm tra chất lượng. Lưu toàn bộ vào `sql/queries.sql`; mỗi truy vấn SELECT được lưu thành `report/table_q*.csv`.

**Kiểm tra**: chọn một đơn vị và 10 dòng, tính tay trung bình trượt rồi so với SQL; không đặc trưng nào dùng giá trị sau thời điểm dự báo.

In [11]:
# SQL statements run one by one in DuckDB; each SELECT prints its first 20 rows and is saved to report/table_q{k}.csv
import duckdb
if 'con' not in globals(): con = duckdb.connect('data/processed/ady.duckdb')   # reuse the connection opened in Step 1
SQL = r"""
-- Step 3: sql/queries.sql
CREATE OR REPLACE TABLE tr AS SELECT * FROM 'data/processed/sf_trees.parquet';
-- Q1: requests per neighborhood and year; seasonal pattern; storm days
SELECT nbhd, year(date) AS yr, SUM(n) AS requests, SUM(hazard) AS hazard FROM tr GROUP BY 1, 2 ORDER BY 1, 2;
SELECT month(date) AS mon, AVG(n) AS n_per_day FROM tr GROUP BY 1 ORDER BY 1;
-- Q2: gust and rain effect: requests on days with gusts above 40 mph versus calm days
SELECT CASE WHEN wsf2 >= 40 THEN 'gust40' WHEN wsf2 >= 25 THEN 'gust25' ELSE 'calm' END AS wind, AVG(n) AS n_per_day, AVG(hazard) AS hazard_per_day FROM tr GROUP BY 1;
-- Q3: features and targets: requests next day and next 7 days
CREATE OR REPLACE TABLE feat AS
SELECT nbhd, date, n, hazard, prcp, wsf2, tmax, dayofweek(date) AS dow, month(date) AS mon,
       LAG(n, 1) OVER w AS n_lag1, AVG(n) OVER (w ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS n_ma7, AVG(n) OVER (w ROWS BETWEEN 27 PRECEDING AND CURRENT ROW) AS n_ma28,
       SUM(prcp) OVER (w ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS prcp3, MAX(wsf2) OVER (w ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS gust3,
       LEAD(n, 1) OVER w AS y_1d, SUM(n) OVER (w ROWS BETWEEN 1 FOLLOWING AND 7 FOLLOWING) AS y_7d
FROM tr WINDOW w AS (PARTITION BY nbhd ORDER BY date);
SELECT COUNT(*) AS n, COUNT(y_7d) AS n7 FROM feat;
"""
k = 0
SQL_NO_COMMENTS = '\n'.join(l for l in SQL.splitlines() if not l.strip().startswith('--'))   # drop comment lines first (they may contain semicolons)
for stmt in SQL_NO_COMMENTS.split(';'):
    stmt = stmt.strip()
    if not stmt: continue
    res = con.execute(stmt)
    if stmt.upper().startswith(('SELECT', 'WITH')):
        k += 1; out = res.df(); out.to_csv(f'report/table_q{k}.csv', index=False); print(f'--- Q{k} ---'); print(out.head(20))
open('sql/queries.sql', 'w').write(SQL)

--- Q1 ---
                     nbhd    yr  requests  hazard
0   Bayview Hunters Point  2015     374.0    52.0
1   Bayview Hunters Point  2016     291.0    31.0
2   Bayview Hunters Point  2017     613.0    95.0
3   Bayview Hunters Point  2018     454.0    40.0
4   Bayview Hunters Point  2019     662.0    88.0
5   Bayview Hunters Point  2020     477.0    36.0
6   Bayview Hunters Point  2021     789.0    80.0
7   Bayview Hunters Point  2022     633.0    14.0
8   Bayview Hunters Point  2023     614.0   204.0
9   Bayview Hunters Point  2024     566.0   149.0
10  Bayview Hunters Point  2025    1199.0   143.0
11  Bayview Hunters Point  2026     574.0    48.0
12         Bernal Heights  2015     354.0    82.0
13         Bernal Heights  2016     277.0    41.0
14         Bernal Heights  2017     449.0    75.0
15         Bernal Heights  2018     459.0    58.0
16         Bernal Heights  2019     502.0    73.0
17         Bernal Heights  2020     389.0    49.0
18         Bernal Heights  2021     586

1332

### 3.2. Tiếp nối SQL: bảng trả lời RQ1 bằng pandas và kiểm định

**Làm gì**: từ kết quả Q1, Q2 dựng bảng có số thứ tự đưa thẳng vào bài; kiểm định khác biệt giữa nhóm (Kruskal-Wallis, không cần giả định chuẩn) và tương quan Spearman giữa mục tiêu và các biến ngoại sinh kèm p-value.

**Đọc thế nào**: p < 0,05 nghĩa là khác biệt giữa nhóm khó do ngẫu nhiên; hệ số Spearman cho chiều và độ mạnh quan hệ đơn điệu; kết luận RQ1 phải là một câu có số, ví dụ "mùa đông cao gấp 1,9 lần mùa hè, p < 0,001".

In [12]:
from scipy.stats import kruskal, spearmanr
q1 = con.execute("SELECT * FROM feat").df()          # feature table created in Q3
if np.issubdtype(q1['date'].dtype, np.number): q1['period'] = q1['date']            # numeric time key (year, cycle)
else: q1['period'] = pd.to_datetime(q1['date'], errors='coerce').dt.month             # datetime key: use month as period
# Table RQ1a: target by period
rq1a = q1.groupby('period')['n'].agg(['mean', 'median', 'std', 'count']).round(3)
rq1a.to_csv('report/table_rq1a_period.csv'); print(rq1a)
groups = [s.values for _, s in q1.groupby('period')['n'] if len(s) > 30]
if len(groups) > 1: print('Kruskal-Wallis across periods:', kruskal(*groups))
# Table RQ1b: Spearman correlation of the target with exogenous variables
ext = [c for c in q1.select_dtypes('number').columns if c not in ('n', 'y_7d') and not c.startswith('n')]
rows = []
for c in ext[:15]:
    r, pv = spearmanr(q1[c], q1['n'], nan_policy='omit'); rows.append([c, round(r, 3), pv])
rq1b = pd.DataFrame(rows, columns=['variable', 'spearman_r', 'p_value']).sort_values('spearman_r', key=abs, ascending=False)
rq1b.to_csv('report/table_rq1b_corr.csv', index=False); print(rq1b)
# Table RQ1c: target by unit, top and bottom 5
rq1c = q1.groupby('nbhd')['n'].agg(['mean', 'count']).sort_values('mean', ascending=False)
pd.concat([rq1c.head(5), rq1c.tail(5)]).round(3).to_csv('report/table_rq1c_units.csv')
print('RQ1 CONCLUSION (fill with real numbers): highest period', rq1a['mean'].idxmax(), 'is', round(rq1a['mean'].max() / max(rq1a['mean'].min(), 1e-9), 2), 'times the lowest period')

         mean  median    std  count
period                             
1       0.757     0.0  1.714  15032
2       0.788     0.0  1.677  13843
3       0.737     0.0  1.666  15190
4       0.692     0.0  1.341  14700
5       0.766     0.0  1.381  15190
6       0.827     0.0  1.484  14700
7       0.810     0.0  1.421  15219
8       0.823     0.0  1.408  15155
9       0.747     0.0  1.360  13892
10      0.829     0.0  1.954  13981
11      0.637     0.0  1.300  13530
12      0.569     0.0  1.576  13981
Kruskal-Wallis across periods: KruskalResult(statistic=np.float64(739.8420068140393), pvalue=np.float64(1.542709309564555e-151))
  variable  spearman_r       p_value
0   hazard       0.426  0.000000e+00
8     y_1d       0.311  0.000000e+00
3     tmax       0.028  1.194792e-30
1     prcp       0.015  1.114990e-09
9   period      -0.014  1.286059e-08
5      mon      -0.014  1.286059e-08
6    prcp3       0.010  3.788617e-05
4      dow      -0.006  7.877093e-03
2     wsf2         NaN           N

## Bước 5 (phần A): bộ hình đa dạng trả lời RQ1

Mỗi hình gắn với một ý của RQ1 và có caption là kết luận. Bộ hình: (1) phân phối mục tiêu, histogram và boxplot; (2) chuỗi theo thời gian của vài đơn vị; (3) heatmap thời kỳ x đơn vị; (4) boxplot mục tiêu theo thời kỳ; (5) scatter mục tiêu với biến ngoại sinh mạnh nhất; (6) heatmap tương quan; (7) so sánh nhóm bằng cột có số. Quy tắc: chữ tiếng Anh, bỏ viền trên và phải, số in đậm trên cột, 300 dpi, tên file `fig_rq1_*.png`.

In [13]:
import seaborn as sns
def style(ax): ax.spines[['top', 'right']].set_visible(False)
d = q1.dropna(subset=['n'])
# (1) distribution
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(d['n'], bins=50, color='#1B6B6D'); ax[0].set_xlabel('n '); ax[0].set_ylabel('Count'); style(ax[0])
ax[1].boxplot(d['n'], vert=False); ax[1].set_xlabel('n'); style(ax[1])
fig.tight_layout(); fig.savefig('report/fig_rq1_distribution.png', dpi=300); plt.close(fig)
# (2) time series of the first 3 units
units = d['nbhd'].unique()[:3]
fig, ax = plt.subplots(figsize=(9, 3))
for u in units:
    s = d[d['nbhd'] == u].sort_values('date'); ax.plot(s['date'], s['n'], lw=0.8, label=str(u))
ax.set_ylabel('n'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('report/fig_rq1_timeseries.png', dpi=300); plt.close(fig)
# (3) heatmap period x unit (top 15 units)
top = d.groupby('nbhd')['n'].mean().nlargest(15).index
hm = d[d['nbhd'].isin(top)].pivot_table(index='nbhd', columns='period', values='n', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 4)); sns.heatmap(hm, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'n'}); ax.set_xlabel('Period'); ax.set_ylabel('Unit')
fig.tight_layout(); fig.savefig('report/fig_rq1_heatmap.png', dpi=300); plt.close(fig)
# (4) boxplot by period
fig, ax = plt.subplots(figsize=(9, 3)); sns.boxplot(data=d, x='period', y='n', color='#9FBFBF', ax=ax, showfliers=False); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_box_period.png', dpi=300); plt.close(fig)
# (5) scatter against the strongest exogenous variable
if len(rq1b):
    v = rq1b.iloc[0].variable
    fig, ax = plt.subplots(figsize=(5, 3.5)); ax.scatter(d[v], d['n'], s=4, alpha=0.3, color='#1B6B6D'); ax.set_xlabel(v); ax.set_ylabel('n'); style(ax)
    fig.tight_layout(); fig.savefig('report/fig_rq1_scatter.png', dpi=300); plt.close(fig)
# (6) correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5)); sns.heatmap(d.select_dtypes('number').corr().round(2), cmap='coolwarm', center=0, ax=ax, annot=False)
fig.tight_layout(); fig.savefig('report/fig_rq1_corr.png', dpi=300); plt.close(fig)
# (7) group comparison with labelled bars
gm = d.groupby('period')['n'].mean()
fig, ax = plt.subplots(figsize=(8, 3)); bars = ax.bar(gm.index.astype(str), gm.values, color='#1B6B6D'); ax.bar_label(bars, fmt='%.1f', fontweight='bold', fontsize=8); ax.set_ylabel('Mean n'); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_period_bar.png', dpi=300); plt.close(fig)
print('7 figures saved to report/. Write each caption as: conclusion + number + condition.')

7 figures saved to report/. Write each caption as: conclusion + number + condition.


## Test case của Notebook_A (chạy trước khi bàn giao)

Mỗi assert là một điều kiện phải đúng; nếu sai, sửa bước tương ứng rồi chạy lại. Sau khi qua hết, ô cuối tạo `manifest.json` để Sinh viên B kiểm tra.

In [14]:
import hashlib, json, os
feat = con.execute("SELECT * FROM feat").df()
assert len(feat) >= 30000, 'fewer than 30k rows, widen the scope'
assert feat['y_7d'].notna().sum() > 0.8 * len(feat), 'too many missing targets'
key = ['nbhd', 'date']
assert not feat.duplicated(key).any(), 'duplicate unit-time keys'
lead_cols = [c for c in feat.columns if c.startswith('y_') and c != 'y_7d']
assert all(c.startswith('y_') for c in lead_cols), 'only target columns may contain future values'
print('Tests A: OK')
feat.to_parquet('data/processed/feat.parquet', index=False)
h = hashlib.md5(open('data/processed/feat.parquet', 'rb').read()).hexdigest()
manifest = {'rows': int(len(feat)), 'cols': list(feat.columns), 'md5': h,
            'time_min': str(feat['date'].min()), 'time_max': str(feat['date'].max()), 'author': 'Student A'}
json.dump(manifest, open('data/processed/manifest.json', 'w'), indent=2, default=str); print(manifest)

Tests A: OK
{'rows': 174413, 'cols': ['nbhd', 'date', 'n', 'hazard', 'prcp', 'wsf2', 'tmax', 'dow', 'mon', 'n_lag1', 'n_ma7', 'n_ma28', 'prcp3', 'gust3', 'y_1d', 'y_7d'], 'md5': '56f8d2e3b9dded46d9e1b0f0425cd83b', 'time_min': '2015-01-01 00:00:00', 'time_max': '2026-09-12 00:00:00', 'author': 'Student A'}


## Lỗi thường gặp và cách xử lý (đọc khi thấy chữ đỏ)

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| ModuleNotFoundError | chưa cài thư viện hoặc chọn sai interpreter | chạy lại pip install ở Bước 0; Ctrl+Shift+P chọn interpreter .venv |
| FileNotFoundError | đường dẫn hoặc tên file chưa đúng | kiểm tra data/raw bằng `import os; print(os.listdir('data/raw'))` |
| KeyError: 'ten_cot' | tên cột thật khác tên trong code | in `df.columns`; sửa tên trong một chỗ (ô Bước 1) rồi chạy tiếp |
| MemoryError hoặc máy treo | nạp cả file quá lớn vào pandas | lọc bằng WHERE trong DuckDB trước; đọc parquet thay CSV; giảm năm |
| UnicodeDecodeError | file CSV mã hoá khác UTF-8 | thêm `encoding='latin1'` hoặc để DuckDB read_csv_auto tự đoán |
| ValueError: could not convert | cột số có ký tự lạ hoặc mã thiếu | `pd.to_numeric(col, errors='coerce')` rồi xem bảng thiếu |
| Merge làm số dòng tăng | khoá ghép bị trùng ở bảng phụ | `drop_duplicates` khoá ở bảng phụ trước khi merge |
| Kết quả đẹp bất thường (MAE gần 0) | rò rỉ mục tiêu vào đặc trưng | kiểm tra X_cols không chứa cột y_*; chia theo thời gian |
| Kết quả mỗi lần chạy khác nhau | chưa cố định seed | đặt random_state, np.random.seed(42) |

Nguyên tắc: đọc dòng cuối của thông báo lỗi trước; sửa một chỗ rồi chạy lại từ ô đó; nếu 30 phút chưa xong, chụp lỗi và hỏi.

## Checklist trước khi báo cáo (đánh dấu [x] khi có file bằng chứng)

- [x] requirements.txt, README.md, .gitignore, kho GitHub có commit trong tuần
- [x] Dữ liệu đúng file cơ quan, trên 30 nghìn dòng: table_describe_raw.csv, profile.html
- [x] EDA: table_describe_numeric.csv, table_describe_categorical.csv, table_eda_notes.csv, table_missing.csv, table_columns.csv, table_target_by_unit.csv, table_target_by_period.csv
- [x] Làm sạch: table_cleaning_log.csv; ghép nguồn thứ hai khớp trên 90%
- [x] SQL: sql/queries.sql, table_q1..q4.csv; RQ1: table_rq1a_period.csv, table_rq1b_corr.csv, table_rq1c_units.csv có p-value
- [x] 7 hình RQ1 trong report/ với caption là câu kết luận có số
- [x] Test case qua; manifest.json bàn giao; người kia đã chạy lại và ghi đối chứng
- [x] (Notebook_B) table_features.csv, table_rq2.csv, 4 hình RQ2, table_tuning.csv, table_rq3_*.csv, table_shap.csv, table_diagnosis.csv, params_best.json
- [x] Audit Log cá nhân có đủ prompt của tuần
- [ ] Báo cáo năm mục đã dán vào kênh nhóm trước 12:00 thứ Tư hoặc thứ Bảy


## Mẫu báo cáo tiến độ (slot 3 thứ Tư và thứ Bảy (12:30 đến 14:45), 7:00 đến 9:15)

Dán vào kênh nhóm trước 12:00 ngày báo cáo, mỗi người một phần. Năm mục, mỗi mục hai đến ba dòng:

1. **Đã xong** từ lần trước: tên file, số dòng, số bảng, số hình (ví dụ: `feat.parquet` 138.204 dòng; 4 bảng RQ1; 7 hình).
2. **Con số chính mới có**: ví dụ MAE naive 6,8; MAE LightGBM 4,7 (5 seed, sd 0,1).
3. **Vướng mắc và đã thử gì**: lỗi cụ thể, ảnh chụp, hai cách đã thử.
4. **Sẽ làm đến lần sau, ai làm**: theo Bảng 13 của đề cương.
5. **Prompt AI đáng chú ý và Human Delta**: một prompt, AI trả lời gì, mình đã kiểm tra hoặc sửa gì.

**Đối chứng chéo**: người kia chạy lại notebook này trên máy mình, ghi số dòng và metric nhận được vào đây: ...............................